# Dimention table creation 


## Customer Dimention Table

In [0]:
from delta.tables import DeltaTable


df_cust = spark.read.table("pysparkdbt.silver.customers")
df_cust = df_cust.select('customer_id', 'full_name', 'city', 'signup_date')


table_name = "pysparkdbt.gold.dim_customer"

if spark.catalog.tableExists(table_name):

    df_cust.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(table_name)

else:

    df_cust.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(table_name)


## Driver Dimention Table

In [0]:
from delta.tables import DeltaTable

df_driv = spark.read.table("pysparkdbt.silver.drivers")
df_driv = df_driv.select('driver_id', 'full_name', 'city', 'driver_rating')


table_name = "pysparkdbt.gold.dim_driver"

if spark.catalog.tableExists(table_name):

    df_driv.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(table_name)

else:

    df_driv.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(table_name)

## Vehicle Dimention Table

In [0]:
from delta.tables import DeltaTable

df_veh = spark.read.table("pysparkdbt.silver.vehicle")
df_veh  = df_veh .select('vehicle_id', 'model', 'make', 'vehicle_type')


table_name = "pysparkdbt.gold.dim_vehicle"

if spark.catalog.tableExists(table_name):

    df_veh.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(table_name)

else:
    df_veh.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(table_name)

## Location Dimention Table

In [0]:
from delta.tables import DeltaTable

df_loc = spark.read.table("pysparkdbt.silver.location")
df_loc  = df_loc .select('location_id', 'city', 'state', 'country')


table_name = "pysparkdbt.gold.dim_location"

if spark.catalog.tableExists(table_name):

    df_loc.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(table_name)

else:
    df_loc.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(table_name)

## Payment Dimention Table

In [0]:
from delta.tables import DeltaTable

df_pay = spark.read.table("pysparkdbt.silver.payment")
df_pay  = df_pay.select('payment_status', 'payment_method')


table_name = "pysparkdbt.gold.dim_payment"

if spark.catalog.tableExists(table_name):

    df_pay.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(table_name)

else:
    df_pay.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(table_name)

# Fact Table

## Trip Fact Table

In [0]:
from pyspark.sql.functions import col, to_date


trip_df = spark.table("pysparkdbt.silver.trip")
driver_df = spark.table("pysparkdbt.silver.drivers")
vehicle_df = spark.table("pysparkdbt.silver.vehicle")  
payment_df = spark.table("pysparkdbt.silver.payment")



fact_trips = (
    trip_df.alias("t")
    
    # Driver join (for rating)
    .join(driver_df.alias("d"), col("t.driver_id") == col("d.driver_id"), "left")
    
    # Vehicle join (for type/model)
    .join(vehicle_df.alias("v"), col("t.vehicle_id") == col("v.vehicle_id"), "left")

    .select(
        col("t.trip_id"),
        col("t.customer_id"),
        col("t.driver_id"),
        to_date(col("t.trip_start_time")).alias("trip_date"),
        
        # from vehicle table
        col("v.model").alias("vehicle_model"),
        col("v.vehicle_type"),
        
        col("t.start_location").alias("pickup_city"),
        col("t.end_location").alias("drop_city"),
        
        col("t.distance_km"),
        col("t.fare_amount"),
        col("t.payment_method"),
        col("t.trip_status"),
        
        # from driver table
        col("d.driver_rating")
    )
)

table_name = "pysparkdbt.gold.fact_trips"

if spark.catalog.tableExists(table_name):

    fact_trips.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(table_name)

else:
    fact_trips.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(table_name)

In [0]:
%sql

select sum(fare_amount) as total_revenue,
 count(trip_id) as total_trips,
avg(distance_km)as avg_trip_distance,
avg(fare_amount) as avg_trip_fare
 from pysparkdbt.gold.fact_trips


In [0]:
%sql
SELECT 
    pickup_city,
    COUNT(*) AS total_trips,
    SUM(fare_amount) AS total_revenue
FROM pysparkdbt.gold.fact_trips
GROUP BY pickup_city
ORDER BY total_revenue DESC;

In [0]:
%sql
SELECT 
    payment_method,
    COUNT(*) AS total_transactions
FROM pysparkdbt.gold.fact_trips
GROUP BY payment_method
ORDER BY total_transactions DESC;

In [0]:
%sql
SELECT 
    driver_id,
    COUNT(*) AS total_trips,
    AVG(driver_rating) AS avg_rating,
    SUM(fare_amount) AS total_revenue
FROM pysparkdbt.gold.fact_trips
GROUP BY driver_id
ORDER BY total_revenue DESC;

In [0]:
%sql
WITH cleaned_data AS (
    SELECT 
        LOWER(TRIM(pickup_city)) AS pickup_city,
        fare_amount
    FROM pysparkdbt.gold.fact_trips
    WHERE pickup_city IS NOT NULL
),

city_revenue AS (
    SELECT 
        pickup_city,
        SUM(fare_amount) AS revenue
    FROM cleaned_data
    GROUP BY pickup_city
),

top3 AS (
    SELECT *
    FROM city_revenue
    ORDER BY revenue DESC
    LIMIT 3
),

total AS (
    SELECT SUM(fare_amount) AS total_revenue 
    FROM cleaned_data
)

SELECT 
    ROUND(
        (SELECT SUM(revenue) FROM top3) / 
        (SELECT total_revenue FROM total) * 100,
    2) AS percentage;

In [0]:
%sql
SELECT 
    payment_method,
    COUNT(CASE WHEN payment_status = 'FAILED' THEN 1 END) * 100.0 / COUNT(*) AS failure_rate
FROM pysparkdbt.silver.payment
GROUP BY payment_method
ORDER BY failure_rate DESC; 

In [0]:
%sql
WITH driver_trips AS (
    SELECT 
        driver_id,
        COUNT(*) AS trip_count
    FROM pysparkdbt.gold.fact_trips
    GROUP BY driver_id
),
ranked AS (
    SELECT *,
           SUM(trip_count) OVER () AS total_trips
    FROM driver_trips
)
SELECT 
    (SUM(trip_count) / MAX(total_trips)) * 100 AS percentage
FROM (
    SELECT *
    FROM ranked
    ORDER BY trip_count DESC
    LIMIT 10
);